# Triển khai hệ thống RAG hoàn chỉnh với Semantic Kernel

Notebook toàn diện này trình bày cách xây dựng một hệ thống Retrieval Augmented Generation (RAG) bằng cách sử dụng Semantic Kernel của Microsoft. Chúng ta sẽ bắt đầu bằng cách chỉ ra những hạn chế của các mô hình AI khi không có quyền truy cập vào dữ liệu cụ thể, sau đó xây dựng một hệ thống RAG hoàn chỉnh với các chiến lược phân đoạn (chunking strategies), cơ sở dữ liệu vector và các phương pháp đánh giá.

## Cài đặt và Thiết lập

Trong mô-đun này, chúng ta sẽ tận dụng Semantic Kernel.

Semantic Kernel là một SDK mã nguồn mở cho phép bạn dễ dàng xây dựng các ứng dụng AI. Nó hỗ trợ C#, Python và Java. Nó sẵn sàng cho sản xuất và nhiều doanh nghiệp lớn đang sử dụng nó.

Nó được thiết kế theo mô-đun, vì vậy bạn có thể dễ dàng thay đổi mô hình mà không cần viết lại toàn bộ codebase của mình.

Các SDK như Semantic Kernel trở nên phổ biến vì bản thân các LLM chỉ có thể xử lý dữ liệu và tạo phản hồi. Chúng không thể truy cập cơ sở dữ liệu của bạn, gọi API của bạn, thực thi mã hoặc tương tác với các hệ thống bên ngoài. Vì vậy, Semantic Kernel quản lý các kết nối đến các dịch vụ AI (như Open AI), cung cấp một hệ thống Plugin nơi bạn có thể viết các hàm mà AI có thể gọi, và quản lý lịch sử và ngữ cảnh cuộc trò chuyện.

Trọng tâm của Semantic Kernel là bộ điều phối Kernel. Trong các ứng dụng AI, bạn cần điều phối nhiều thành phần chuyển động như dịch vụ AI, cơ sở dữ liệu, API, hệ thống ghi nhật ký, v.v. Kernel là bộ điều phối trung tâm giữ tất cả những điều này lại với nhau. Nó chứa các Dịch vụ (như Dịch vụ AI, dịch vụ đăng nhập, dịch vụ xác thực) và Plugins (các hàm tùy chỉnh mà AI có thể gọi, như truy cập cơ sở dữ liệu của bạn). Hãy xem xét một kịch bản doanh nghiệp thực tế, nơi một trợ lý AI cần truy vấn CRM, kiểm tra mức độ tồn kho, tạo đề xuất và ghi nhật ký tất cả các tương tác để tuân thủ. Nếu không có kernel, mỗi đoạn mã sẽ cần biết cách kết nối với tất cả các dịch vụ này. Với kernel, mọi thứ được cấu hình một lần. Bởi vì tất cả các hoạt động AI đều thông qua kernel, bạn có một điểm kiểm soát duy nhất để ghi nhật ký và quản lý. Kernel hiện hỗ trợ MCP, bao bọc kernel trong một máy chủ nhận biết mạng giao tiếp ngôn ngữ MCP để kernel (hoặc agent) này có thể được những người khác phát hiện.

Các thành phần của Semantic Kernel:

1. AI Service Connectors: Tất nhiên chúng ta đang sống trong một thế giới mà chúng ta sử dụng nhiều mô hình AI, và các mô hình này sử dụng các API và phương thức xác thực khác nhau. Một connector trong SK là một lớp trừu tượng để ngăn chặn sự phụ thuộc vào nhà cung cấp (cho phép bạn thay đổi giữa nhiều mô hình). Ví dụ, AzureChatCompletion là một Service, GoogleAIChatCompletion là một Service khác. Kernel chịu trách nhiệm gọi các connector này.
2. Vector Store Connectors: Đây là cốt lõi của RAG. Đây là cầu nối giữa các vector stores và kernel.
3. Functions and Plugins: Plugins là thứ cho phép bạn cấp quyền cho LLM truy cập các công cụ. Một function là một khả năng duy nhất bạn cung cấp cho LLM (ví dụ: một hàm python) và một plugin là một nhóm các hàm liên quan (DatabasePlugin có thể chứa các hàm như GetUser, UpdateUser, GetOrders).
4. Prompt Templates: Viết các prompt hiệu quả với các chuỗi đa dòng bên trong mã của chúng ta có thể trở nên lộn xộn và khó bảo trì. Đồng thời, những người không phải là nhà phát triển như Prompt Engineer cũng không thể làm việc với nó. Thực tế, nó là một tệp văn bản hoặc một chuỗi kết hợp các hướng dẫn tĩnh với các placeholder động. Hướng dẫn tĩnh có thể là: “Bạn là một nhà phân tích tài chính chuyên nghiệp, tóm tắt báo cáo sau” và các placeholder động có thể là user_input, hoặc một hàm để lấy giá cổ phiếu.
5. Filters: Đoạn mã chặn việc thực thi kernel tại các thời điểm quan trọng, như trước khi một hàm được gọi hoặc sau khi một prompt được render. Bạn có thể làm điều này để lọc dữ liệu PII (đảm bảo không có số thẻ tín dụng người dùng nào được gửi đến LLM bên ngoài).

**Đầu tiên, cài đặt các gói cần thiết:**

In [ ]:
# Run this cell first to install all required packages
!pip install semantic-kernel openai numpy scikit-learn faiss-cpu

## Thiết lập môi trường

In [ ]:
from typing import List, Dict
import numpy as np
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv()


# Semantic Kernel core imports (verified working June 2025)
import semantic_kernel as sk
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAITextEmbedding
from semantic_kernel.contents import ChatHistory
from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings


# For vector storage - we'll build our own simple system
import faiss
from sklearn.metrics.pairwise import cosine_similarity

print("Semantic Kernel environment setup complete!")

## Tạo Mô hình tài liệu và Kho lưu trữ Vector của chúng ta

Chúng ta sẽ sử dụng cơ sở dữ liệu vector FAISS. FAISS là một thư viện cơ sở dữ liệu vector cục bộ chạy hoàn toàn trên máy của bạn. Trong các kịch bản sản xuất, chúng ta thường sử dụng cơ sở dữ liệu vector đám mây như Azure Search. Nhưng các khái niệm đều giống nhau, một cơ sở dữ liệu vector cho phép chúng ta lưu trữ các vector trong đó.

Giải thích mã:

1. Chúng ta sẽ tạo một lớp (để tạo một đối tượng từ đó, tức là một phiên bản của lớp) để biểu diễn các đoạn tài liệu (document chunks). Sẽ có các bình luận chi tiết để ghi lại những gì sẽ có bên trong các đối tượng chunk này.
2. Sau đó, chúng ta sẽ tạo một lớp khác chứa các hàm (phương thức) cho phép chúng ta tìm kiếm các chunk này bằng cách sử dụng vector search (thay vì khớp chính xác từng từ).
    - Hàm khởi tạo: Một hàm để khởi tạo vector store của chúng ta. Chúng ta sẽ khởi tạo vector store của mình với các thuộc tính này:
        - Embedding Dimension: Số lượng số mà mỗi đoạn tài liệu sẽ có khi được chuyển đổi thành vector. Mô hình text embedding của OpenAI (mà chúng ta sẽ sử dụng) chuyển đổi bất kỳ văn bản nào thành chính xác 1536 số. Vì vậy, “Hello world” trở thành [0.123, -0.456, 0.789, … tổng cộng 1536 số). Mỗi đoạn văn bản đều nhận được chính xác 1536 số, cho dù đó là một từ hay một đoạn văn.
        - Index: Hãy nghĩ về một index như mục lục, thay vì tìm kiếm mọi hàng trong cơ sở dữ liệu để tìm “John Smith”, index cho bạn biết John ở hàng nào. Trong cơ sở dữ liệu vector, một index là một cấu trúc dữ liệu tổ chức các vector để bạn có thể nhanh chóng tìm thấy những vector tương tự. Nếu không có index, việc tìm kiếm các vector tương tự sẽ yêu cầu so sánh truy vấn của bạn với mọi vector được lưu trữ (từng cái một). Với một index, FAISS tổ chức trước các vector để nó có thể nhanh chóng chuyển đến những vector tương tự nhất. Chúng ta đang khởi tạo index đó.
        - Documents: Lưu trữ các đoạn tài liệu thực tế (văn bản gốc cộng với metadata) trong một danh sách Python thông thường.
        - ID to Index: Đây là một dictionary ánh xạ ID tài liệu đến vị trí của chúng trong danh sách của chúng ta. Vì vậy, chúng ta có thể nhanh chóng tìm đoạn tài liệu 5 chẳng hạn mà không cần tìm kiếm toàn bộ danh sách.
    - Hàm thêm tài liệu: Hàm này nhận một loạt các đoạn tài liệu và lưu trữ chúng trong vector store của chúng ta. Chúng ta xử lý từng tài liệu (đã được vectorize thông qua mô hình embedding của chúng ta) và sau đó thêm chúng vào FAISS index để tìm kiếm sự tương đồng nhanh chóng. Chúng ta giữ văn bản gốc và metadata trong danh sách documents của mình để có thể truy xuất sau này.
    - Hàm tìm kiếm: Hàm này nhận câu hỏi của người dùng (được chuyển đổi thành vector 1536 số) và tìm các đoạn tài liệu tương tự nhất.
3. Chúng ta hiện có vector store tùy chỉnh của mình. Bây giờ chúng ta cần kết nối một LLM với nó và đây là lúc Semantic Kernel xuất hiện. Chúng ta sẽ sử dụng Semantic Kernel để khai thác các khả năng embedding (chuyển đổi văn bản thành vector) và các khả năng chat completion (công cụ suy luận LLM).

In [ ]:


@dataclass
# We will create a DocumentChunk class to represent each piece of a document
class DocumentChunk:

   # Required fields - every chunk must have these
   id: str                    # Unique name for this chunk, like "policy_doc_chunk_1"
   content: str              # The actual text content of this chunk 
   source_doc_id: str        # Which original document this came from
   title: str                # Human-readable title of the original document
   chunk_index: int          # Which piece is this? (0=first chunk, 1=second, etc.)
   
   # Optional fields - these have default values
   department: str = ""      # Which team owns this document (optional)
   doc_type: str = ""        # What kind of doc is this - policy, guide, etc. (optional)
   embedding: List[float] = None  # The vector representation (list of numbers) for this text

# This is a class with methods to search for documents. Instead of exact word matches, it finds documents with similar meanings. 
class SimpleVectorStore:
  
   
   # This is a function to initialize our vector store. We will use FAISS, where we can store and search through many document chunks quickly. 
   def __init__(self, embedding_dimension: int = 1536):
       # How many numbers are in each embedding vector? What this means is that each document chunk will be represented by a list of 1536 numbers capturing its meaning.
       # OpenAI's text-embedding-ada-002 model (if we use it) gives us 1536 numbers for each piece of text.
       self.embedding_dimension = embedding_dimension
       
       # Create a FAISS index to store our document embeddings. Index FlatIP means we will use inner product similarity (like cosine similarity) to find similar documents.
       self.index = faiss.IndexFlatIP(embedding_dimension)
       
       # Store the actual document chunks (the text and metadata) in a list called documents
       self.documents: List[DocumentChunk] = []
       
       # This dictionary maps document IDs to their position in the documents list, so we can quickly find a document by its ID. for example, if we have a document with ID "policy_doc_chunk_1", we can find it in our list of documents by looking up "policy_doc_chunk_1" in this dictionary.
       # This is like a quick lookup table - if we have a document with ID "policy_doc_chunk_1", we can find it in our list of documents by looking up "policy_doc_chunk_1" in this dictionary.
       self.id_to_index = {}
   
   #This function adds a batch of documents to our vector store.
   def add_documents(self, documents: List[DocumentChunk]):
      
       embeddings = []  # This is where we will store the vector representations (embeddings) of each document chunk
       
       # Process each document one by one
       for doc in documents:
           # Every document must have its embedding (vector representation) already calculated
           if doc.embedding is None:
               raise ValueError(f"Document {doc.id} missing embedding")
           
           # CRITICAL: Normalize the embedding vector
           # Why? So we can use cosine similarity (comparing angles, not lengths)
           # Think of vectors as arrows - we want to compare which direction they point
           embedding_array = np.array(doc.embedding)  # Convert list to numpy array for math
           normalized_embedding = embedding_array / np.linalg.norm(embedding_array)  # Make length = 1
           embeddings.append(normalized_embedding) # This is the normalized vector representation of the document chunk stored in our embeddings list
           
           # These are the metadata fields we will use to identify and retrieve the document later
           doc_index = len(self.documents)           # What position will this doc be at?
           self.documents.append(doc)                # Add document to our storage
           self.id_to_index[doc.id] = doc_index     # Remember: this ID is at this position
       
       # Add all the normalized vectors to FAISS for lightning-fast search
       embeddings_array = np.array(embeddings).astype('float32')  # FAISS requires float32 type
       self.index.add(embeddings_array)
       
       print(f"Added {len(documents)} documents to vector store")
   
   # This function searches for documents similar to a user's question. 
   def search(self, query_embedding: List[float], k: int = 3, score_threshold: float = 0.0):
       """
       Find documents most similar to a query
       
       How it works:
       1. Take the query's embedding (vector representation)
       2. Compare it to all stored document embeddings
       3. Return the k most similar ones above the threshold
       
       Args:
           query_embedding: The vector representation of user's question
           k: How many results to return (top 3, top 5, etc.)
           score_threshold: Only return docs with similarity above this score
       
       Returns:
           List of (document, similarity_score) pairs, sorted by similarity
       """
       # Edge case: if no documents stored, return empty
       if self.index.ntotal == 0:
           return []
       
       # Normalize the query embedding just like we did for stored documents
       # This ensures fair comparison - we're comparing directions, not magnitudes
       query_array = np.array(query_embedding)
       normalized_query = query_array / np.linalg.norm(query_array)
       
       # Ask FAISS to find the k most similar vectors
       # reshape(1, -1) because FAISS expects 2D array (rows=queries, cols=dimensions)
       # Even though we only have 1 query, we need to format it as [[1, 2, 3, ...]]
       scores, indices = self.index.search(normalized_query.reshape(1, -1).astype('float32'), k)
       
       # Convert FAISS results into our format
       results = []
       for score, idx in zip(scores[0], indices[0]):  # [0] because we only sent 1 query
           # FAISS returns -1 if it runs out of documents before reaching k
           # Also filter by minimum similarity score
           if idx >= 0 and score >= score_threshold:
               # Use the index to get the actual document from our storage
               document = self.documents[idx]
               results.append((document, float(score)))
       
       return results

# Set up the AI services we'll use
kernel = Kernel()  # Semantic Kernel is our AI orchestration framework

# Service 1: Chat completion (generates responses to questions)
chat_service = OpenAIChatCompletion(
   ai_model_id="gpt-3.5-turbo"  # Which OpenAI model to use for chat
)
kernel.add_service(chat_service)  # Register this service with the kernel. This allows us to use the chat service in our semantic kernel for generating responses to user queries.

# Service 2: Text embedding (converts text into vector representations)
embedding_service = OpenAITextEmbedding(
   ai_model_id="text-embedding-ada-002"  # OpenAI's embedding model
)
kernel.add_service(embedding_service)  # Register this service with the kernel

#Now our kernel has both chat and embedding services ready to use. So we can ask questions and get answers, as well as convert text into vectors for similarity search.
# We also set up a simple vector store using FAISS to store and search through document chunks based on their meanings.

print("Semantic Kernel initialized with OpenAI services")
print("Using simple vector store with FAISS for document storage")

# Phần 1: Minh họa vấn đề - Không có quyền truy cập vào dữ liệu riêng tư

Hãy bắt đầu bằng cách chỉ ra điều gì sẽ xảy ra khi chúng ta hỏi một mô hình AI về thông tin mà nó chưa được huấn luyện.

Đoạn mã dưới đây rất đơn giản, chúng ta có một loạt các "tài liệu" được lưu trữ trong một mảng. Chúng ta sẽ đặt câu hỏi về từng tài liệu này (nhưng chúng ta sẽ không thực sự triển khai một hệ thống RAG phù hợp) vì vậy chúng ta nên mong đợi mô hình sẽ không biết chúng ta đang nói về điều gì.

In [ ]:
# Sample company data that the model wouldn't know about.
# This represents the private, "ground-truth" information.
company_documents = [
    {
        "id": "product_001",
        "title": "CloudSync Pro Enterprise Plan",
        "content": """CloudSync Pro Enterprise offers unlimited storage, advanced encryption, 
        real-time collaboration for up to 500 users, priority support, and custom integrations. 
        Pricing: $49/month per user with annual commitment. Features include: automatic backup, 
        version control, audit logs, SSO integration, and 99.9% uptime SLA.""",
        "metadata": {"department": "product", "type": "pricing"}
    },
    {
        "id": "policy_001", 
        "title": "Remote Work Policy 2024",
        "content": """Effective January 2024: All employees may work remotely up to 3 days per week. 
        Remote work requires approval from direct manager. Equipment stipend of $500 annually 
        for home office setup. Mandatory video calls for team meetings. Core hours: 10 AM - 3 PM 
        local time for collaboration.""",
        "metadata": {"department": "hr", "type": "policy"}
    },
    {
        "id": "process_001",
        "title": "Customer Refund Process",
        "content": """Step 1: Customer submits refund request through support portal. 
        Step 2: Support agent reviews within 24 hours. Step 3: For amounts under $100, 
        automatic approval. Step 4: For amounts over $100, requires manager approval. 
        Step 5: Refunds processed within 3-5 business days to original payment method. 
        Full refunds available within 30 days of purchase.""",
        "metadata": {"department": "support", "type": "process"}
    },
    {
        "id": "guide_001",
        "title": "New Employee Onboarding Checklist",
        "content": """Day 1: IT setup and system access. Day 2: Department orientation and mentor assignment. 
        Week 1: Complete mandatory training modules (security, compliance, company culture). 
        Week 2: Shadow team members and review project documentation. Month 1: Complete 
        probationary review and set 90-day goals.""",
        "metadata": {"department": "hr", "type": "guide"}
    }
]

# The questions we want to test against the model's base knowledge.
test_questions = [
    "What is the pricing for CloudSync Pro Enterprise?",
    "How many days per week can employees work remotely?",
    "What is the refund approval process for purchases over $100?",
    "What happens during the first week of employee onboarding?"
]


async def run_direct_to_model_test():
    """
    Tests questions directly against the base AI model to demonstrate
    its lack of knowledge about our private company data.
    """
    print("TESTING MODEL WITHOUT RAG - Questions about private company data:")
    print("=" * 70)

    chat_service = kernel.get_service(type=OpenAIChatCompletion)

    # ***FIX 1: Create a default settings object for OpenAI chat models.***
    execution_settings = OpenAIChatPromptExecutionSettings()

    for i, question in enumerate(test_questions, 1):
        print(f"\nQuestion {i}: {question}")
        
        chat_history = ChatHistory()
        chat_history.add_user_message(question)

        # ***FIX 2: Pass the 'settings' object into the function call.***
        response = await chat_service.get_chat_message_content(
            chat_history=chat_history,
            settings=execution_settings  # This argument is now required
        )
        
        print(f"Model Response: {str(response)}")
        print("-" * 50)


# NOTE: This code assumes your Kernel is initialized and the OpenAI API key 
# is configured in your environment before running.

print("✅ Starting test...")
# Run the single, simplified test function.
await run_direct_to_model_test()

## Những gì chúng ta vừa quan sát

Mô hình này hoặc là:
1. **Không thể trả lời** vì không có quyền truy cập vào thông tin cụ thể của công ty
2. **Cung cấp các phản hồi chung chung** có thể không khớp với các chính sách thực tế của bạn
3. **Đưa ra các giả định** có thể không chính xác đối với ngữ cảnh cụ thể của bạn

Đây chính xác là lý do tại sao chúng ta cần RAG - để cấp cho mô hình quyền truy cập vào dữ liệu cụ thể của bạn trong khi vẫn giữ được khả năng suy luận của nó.

---

# Phần 2: Các chiến lược Document Chunking

In [ ]:
# In this section, we will explore different text chunking strategies. 
# We can either do a simple character-based split or a more semantic split that respects paragraphs and sentences. For example, on the latter, we will try to keep sentences together and avoid breaking them in the middle.

# At a high level, what this function does is take a long piece of text and break it into smaller pieces (chunks) that are easier to work with.
def simple_text_splitter(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Simple character-based text splitter with overlap
    """
    chunks = [] # This will hold our text chunks
    start = 0 # Starting position in the text
    
    #Loop until we reach the end of the text.
    while start < len(text):
        # Calculate where this chunk should end
        end = min(start + chunk_size, len(text))
        
        # Try to end at a sentence boundary (but only if we're not at the very end)
        if end < len(text):
            last_period = text.rfind('.', start, end)
            if last_period > start + chunk_size // 2:
                end = last_period + 1
        
        # Extract the chunk
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        
        # Move to next position
        # FIXED: Ensure we always move forward, even with large overlap
        next_start = end - overlap
        start = max(next_start, start + 1)  # Always move at least 1 character forward
        
        # If we've reached the end, break
        if end >= len(text):
            break
    
    return chunks

# What this function does is take a long piece of text and break it into smaller pieces (chunks) that respect paragraph and sentence boundaries. So instead of just cutting it off at a certain number of characters, it tries to keep whole sentences together and avoid breaking them in the middle. This is useful because it helps preserve the meaning of the text and makes it easier to understand.
def semantic_text_splitter(text: str, max_chunk_size: int = 400) -> List[str]:
    """
    Split text respecting paragraph and sentence boundaries
    """
    # Split by paragraphs first (handle both \n\n and single \n)
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    
    chunks = []
    current_chunk = ""
    
    for paragraph in paragraphs:
        # If this paragraph alone is too big, split it by sentences
        if len(paragraph) > max_chunk_size:
            sentences = [s.strip() for s in paragraph.split('.') if s.strip()]
            
            for sentence in sentences:
                sentence_with_period = sentence + '.' if not sentence.endswith('.') else sentence
                
                # Check if adding this sentence would exceed our limit
                if current_chunk and len(current_chunk) + len(sentence_with_period) + 1 > max_chunk_size:
                    chunks.append(current_chunk.strip())
                    current_chunk = sentence_with_period
                else:
                    current_chunk += " " + sentence_with_period if current_chunk else sentence_with_period
        else:
            # Try to add the whole paragraph
            if current_chunk and len(current_chunk) + len(paragraph) + 2 > max_chunk_size:
                chunks.append(current_chunk.strip())
                current_chunk = paragraph
            else:
                current_chunk += "\n" + paragraph if current_chunk else paragraph
    
    # Don't forget the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks

# Test different chunking strategies
sample_doc = company_documents[0]
print("CHUNKING STRATEGY COMPARISON:")
print("=" * 35)

print(f"Original document: {sample_doc['title']}")
print(f"Length: {len(sample_doc['content'])} characters")

# Test simple chunking with safer parameters
print("\n1. SIMPLE CHARACTER-BASED CHUNKING:")
simple_chunks = simple_text_splitter(sample_doc['content'], chunk_size=200, overlap=30)  # Reduced overlap
for i, chunk in enumerate(simple_chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}")

# Test semantic chunking
print("\n2. SEMANTIC CHUNKING (respects paragraphs):")
semantic_chunks = semantic_text_splitter(sample_doc['content'], max_chunk_size=250)
for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}")

print("\nTRADE-OFFS:")
print("- Simple chunking: Predictable sizes, may break mid-sentence")
print("- Semantic chunking: Preserves meaning, variable sizes")

## Kiểm tra Toàn bộ Pipeline RAG

Bây giờ chúng ta sẽ triển khai một hệ thống RAG đơn giản sử dụng Semantic Kernel để xử lý việc truy xuất tài liệu và tạo câu trả lời.

1. semantic_chunker: Nhiệm vụ của nó là lấy một chuỗi văn bản và chia nó thành các chuỗi nhỏ hơn (chunks). Nó trước tiên chia văn bản theo đoạn văn bất cứ khi nào nó thấy hai dấu xuống dòng (\n\n) - điều này đảm bảo rằng các câu thuộc về nhau sẽ ở cùng nhau. Sau đó, nó lặp lại qua các đoạn văn này để nhóm chúng thành một chunk theo kích thước chunk tối đa. Bằng cách tôn trọng các điểm ngắt tự nhiên trong văn bản, nó đảm bảo rằng các câu liên quan ở cùng nhau, tạo ra các chunk thông tin tập trung, chất lượng cao. Điều này làm tăng đáng kể tỷ lệ "tín hiệu trên nhiễu" của dữ liệu của chúng ta.
2. ingest_documents_semantic: Hàm này giải quyết vấn đề lớn đầu tiên của bất kỳ hệ thống RAG nào. Chuẩn bị dữ liệu cho AI. Nó chấp nhận một danh sách các tài liệu, một vector)store (cơ sở dữ liệu tùy chỉnh mà chúng ta đã xây dựng trước đó), và embedding_service. Nó lấy các tài liệu này và chuyển đổi chúng thành các vector. Nó lặp lại qua từng tài liệu, gọi embedding_service và tạo một đối tượng DocumentChunk (với vector, văn bản gốc và metadata).
3. ask_with_semantic_rag: Động cơ cốt lõi của RAG. Nó chấp nhận câu hỏi của người dùng, kernel và vector_store (cơ sở tri thức của chúng ta). Nó chuyển câu hỏi của người dùng thông qua một embedding service để lấy biểu diễn vector, sau đó sử dụng phương pháp tìm kiếm vector để tìm các chunk tài liệu có vector gần nhất. Sau đó, chúng ta tăng cường Prompt và cuối cùng tạo câu trả lời.

In [ ]:
# --- Helper Function for Semantic Chunking ---

def semantic_chunker(text: str, max_chunk_size: int = 300) -> List[str]:
    """
    Splits text into chunks, respecting paragraph boundaries to keep related sentences together.
    
    This is a pure utility function; it doesn't need any AI services or state.
    Its only job is to intelligently split text based on structure.
    """
    # First, split the text into paragraphs based on double newlines.
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    
    chunks = []
    current_chunk = ""
    
    # Iterate through each paragraph to build chunks up to the max size.
    for paragraph in paragraphs:
        # If adding the next paragraph would make the current chunk too large...
        if current_chunk and (len(current_chunk) + len(paragraph) + 2) > max_chunk_size:
            # ...finalize the current chunk...
            chunks.append(current_chunk)
            # ...and start a new chunk with the current paragraph.
            current_chunk = paragraph
        else:
            # Otherwise, add the paragraph to the current chunk.
            current_chunk += ("\n\n" + paragraph) if current_chunk else paragraph
            
    # Add the last remaining chunk to the list.
    if current_chunk:
        chunks.append(current_chunk)
        
    return chunks

# --- Core RAG Functions ---

async def ingest_documents_semantic(
    documents: List[Dict], 
    vector_store: SimpleVectorStore, 
    embedding_service: OpenAITextEmbedding
) -> None:
    """
    Processes and ingests documents using the semantic chunking strategy.
    """
    print(f"Ingesting {len(documents)} documents with semantic chunking...")
    all_chunks_to_add = []
    
    for doc in documents:
        # Use our standalone helper function to get semantically coherent chunks.
        text_chunks = semantic_chunker(doc["content"])
        
        for i, chunk_text in enumerate(text_chunks):
            # Skip chunks that are too short to have meaningful content.
            if len(chunk_text) < 20:
                continue

            # Generate the embedding vector for the chunk's content.
            embedding = (await embedding_service.generate_embeddings([chunk_text]))[0]
            
            # Create the DocumentChunk object.
            chunk = DocumentChunk(
                id=f"{doc['id']}_chunk_{i}",
                content=chunk_text,
                source_doc_id=doc["id"],
                title=doc["title"],
                chunk_index=i,
                embedding=embedding
            )
            all_chunks_to_add.append(chunk)

    vector_store.add_documents(all_chunks_to_add)
    print(f"Added {len(all_chunks_to_add)} new chunks to the vector store.")


async def ask_with_semantic_rag(
    question: str, 
    kernel: Kernel, 
    vector_store: SimpleVectorStore
) -> str:
    """
    Asks a question using the RAG pattern with the semantically chunked documents.
    """
    # Get the necessary AI services from the kernel.
    embedding_service = kernel.get_service(type=OpenAITextEmbedding)
    chat_service = kernel.get_service(type=OpenAIChatCompletion)
    
    # 1. RETRIEVE: Convert the question to an embedding and search the vector store.
    query_embedding = (await embedding_service.generate_embeddings([question]))[0]
    search_results = vector_store.search(query_embedding, k=3, score_threshold=0.3)
    
    if not search_results:
        return "I could not find any relevant information in the documents to answer that question."
        
    # 2. AUGMENT: Build the context string from the retrieved document chunks.
    context = "\n\n---\n\n".join([result.content for result, score in search_results])
    
    # Create the final prompt that instructs the AI and provides the context.
    prompt = f"""
Answer the following question based ONLY on the context provided below.

CONTEXT:
---
{context}
---

QUESTION: {question}

ANSWER:
"""

    # 3. GENERATE: Send the augmented prompt to the chat model to get the final answer.
    chat_history = ChatHistory()
    chat_history.add_user_message(prompt)
    
    # Define execution settings for the AI call.
    settings = OpenAIChatPromptExecutionSettings(max_tokens=200, temperature=0.1)
    
    response = await chat_service.get_chat_message_content(chat_history, settings)
    
    return str(response)

# --- Main Execution Block ---

# 1. Initialize our vector store for this RAG process.
semantic_vector_store = SimpleVectorStore()

# 2. Get the embedding service from the kernel, as it's needed for ingestion.
embedding_service = kernel.get_service(type=OpenAITextEmbedding)

# 3. Call the ingestion function to process documents and populate the vector store.
await ingest_documents_semantic(company_documents, semantic_vector_store, embedding_service)

# 4. Ask a question using the populated vector store.
print("\n" + "="*50)
print("TESTING RAG SYSTEM WITH SEMANTIC CHUNKING:")
question_to_ask = "What is the pricing for CloudSync Pro Enterprise?"
answer = await ask_with_semantic_rag(question_to_ask, kernel, semantic_vector_store)

print(f"\nQ: {question_to_ask}")
print(f"A: {answer}")

# Phần 4: Cấu hình và điều chỉnh nâng cao

Chúng ta đang thử nghiệm hai cách đơn giản để hệ thống RAG của chúng ta đưa ra câu trả lời tốt hơn – thứ nhất, bằng cách thay đổi cách chúng ta yêu cầu AI phản hồi (kiểu thân thiện so với kiểu chuyên nghiệp), và thứ hai, bằng cách điều chỉnh mức độ kỹ tính của chúng ta về việc tài liệu nào được đưa vào (khớp chính xác so với khớp lỏng lẻo).

Cách thứ hai được gọi là similarity threshold. Similarity threshold giống như việc đặt ra tiêu chuẩn cho các kết quả tìm kiếm "liên quan". Đó là một con số nằm giữa 0 và 1, xác định mức độ tương đồng của một đoạn tài liệu với câu hỏi của bạn trước khi chúng ta đưa nó vào câu trả lời.

Khi ngưỡng quá thấp: Nếu bạn hỏi "Chính sách nghỉ phép của chúng ta là gì?" với ngưỡng 0.2, bạn có thể nhận được kết quả về chính sách nghỉ phép, phúc lợi nhân viên, theo dõi thời gian và các ngày lễ của công ty. Mặc dù tất cả đều liên quan đến HR, nhưng điều này khiến người dùng bị ngập trong thông tin không trực tiếp trả lời câu hỏi của họ. AI sau đó phải sàng lọc tất cả bối cảnh bổ sung này, có khả năng làm giảm chất lượng của câu trả lời cuối cùng.

Khi ngưỡng quá cao: Nếu bạn hỏi "Làm cách nào để yêu cầu nghỉ phép?" với ngưỡng 0.7, bạn có thể không nhận được bất kỳ kết quả nào vì không có tài liệu nào chứa cụm từ chính xác đó, mặc dù tài liệu chính sách nghỉ phép của bạn giải thích rõ ràng quy trình. Người dùng cuối cùng thất vọng với các phản hồi "không tìm thấy thông tin" khi câu trả lời thực sự tồn tại trong cơ sở kiến thức của bạn.

Tìm điểm ngọt: Mục tiêu là tìm một ngưỡng cung cấp đủ thông tin liên quan mà không bao gồm nhiễu. Đối với hầu hết các tài liệu doanh nghiệp, ngưỡng từ 0.3-0.5 hoạt động tốt – đủ cao để lọc ra nội dung không liên quan, nhưng đủ thấp để nắm bắt thông tin liên quan có thể sử dụng cách diễn đạt khác với câu hỏi của bạn.

In [ ]:
class OptimizedRAG(SimpleRAG):
    """Simple RAG with optimization features"""
    
    async def ask_with_custom_prompt(self, question, prompt_template):
        """Ask question with a custom prompt"""
        # Search for relevant chunks
        query_embedding = await self.embedding_service.generate_embeddings([question])
        results = self.vector_store.search(query_embedding[0], k=3, score_threshold=0.3)
        
        if not results:
            return "No relevant information found."
        
        # Build context
        context = "\n".join([doc.content for doc, score in results])
        
        # Use custom prompt template
        prompt = prompt_template.format(context=context, question=question)
        
        # Generate answer
        from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
        
        chat_history = ChatHistory()
        chat_history.add_user_message(prompt)
        settings = OpenAIChatPromptExecutionSettings(max_tokens=200, temperature=0.1)
        
        response = await self.chat_service.get_chat_message_content(chat_history, settings)
        return str(response)
    
    async def test_thresholds(self, query, thresholds=[0.1, 0.3, 0.5, 0.7]):
        """Test different similarity thresholds"""
        query_embedding = await self.embedding_service.generate_embeddings([query])
        
        print(f"Testing thresholds for: '{query}'")
        
        for threshold in thresholds:
            results = self.vector_store.search(query_embedding[0], k=5, score_threshold=threshold)
            
            print(f"\nThreshold {threshold}: {len(results)} results")
            if results:
                scores = [score for _, score in results]
                print(f"  Score range: {min(scores):.3f} - {max(scores):.3f}")
                print(f"  Documents: {', '.join([doc.title for doc, _ in results[:2]])}")

# Create optimized RAG system
opt_rag = OptimizedRAG(kernel)
await opt_rag.add_documents(company_documents)

# Test custom prompts
customer_prompt = """You are a helpful customer service agent. 

Context: {context}

Customer question: {question}

Friendly response:"""

employee_prompt = """You are an internal HR assistant.

Context: {context}

Employee question: {question}

Professional response:"""

# Test different prompt styles
question = "What is our remote work policy?"

print("CUSTOMER SERVICE STYLE:")
customer_answer = await opt_rag.ask_with_custom_prompt(question, customer_prompt)
print(customer_answer)

print("\nHR ASSISTANT STYLE:")
hr_answer = await opt_rag.ask_with_custom_prompt(question, employee_prompt)
print(hr_answer)

# Test similarity thresholds
print("\n" + "="*40)
await opt_rag.test_thresholds("employee remote work")

## Tóm tắt các thực hành tốt nhất

### Xử lý tài liệu
- **Sử dụng phân đoạn ngữ nghĩa (semantic chunking)** tôn trọng ranh giới đoạn văn và câu
- **Kích thước đoạn tối ưu: 300-400 ký tự** cho hầu hết các tài liệu kinh doanh
- **Bao gồm sự chồng chéo có ý nghĩa** (50-80 ký tự) để duy trì ngữ cảnh
- **Bảo tồn siêu dữ liệu phong phú (rich metadata)** để lọc và ghi nhận nguồn

### Cấu hình tìm kiếm vector (Vector Search Configuration)
- **Bắt đầu với FAISS** cho phát triển cục bộ và sản xuất quy mô nhỏ
- **Sử dụng ngưỡng tương đồng (similarity thresholds)** khoảng 0.3 để cân bằng độ chính xác/độ phủ
- **Truy xuất 3-5 tài liệu** để cung cấp ngữ cảnh đầy đủ mà không gây nhiễu
- **Chuẩn hóa Embedding** để tính toán độ tương đồng nhất quán

### Prompt Engineering
- **Tạo các hướng dẫn cụ thể theo vai trò (role-specific prompts)** cho các loại người dùng khác nhau (khách hàng, nhân viên, quản lý điều hành)
- **Bao gồm các hướng dẫn rõ ràng** để xử lý các trường hợp không có thông tin
- **Sử dụng các mẫu có cấu trúc (structured templates)** phân tách ngữ cảnh khỏi câu hỏi
- **Kiểm tra các biến thể Prompt** để tối ưu hóa cho các trường hợp sử dụng cụ thể của bạn

## Các bước tiếp theo

1. **Bắt đầu với chức năng cốt lõi** - Vận hành RAG cơ bản với tài liệu của bạn
2. **Thêm giám sát sớm** - Triển khai thu thập nhật ký và số liệu
3. **Tùy chỉnh cho miền của bạn** - Điều chỉnh prompt và phân đoạn cho nội dung của bạn
4. **Lặp lại dựa trên phản hồi** - Sử dụng tương tác người dùng thực để cải thiện hệ thống
5. **Lên kế hoạch sản xuất** - Xem xét khả năng mở rộng, giám sát và bảo trì